# Cellular Automata — Reproducing paper Tables 7–10 with NumPy

Bhuvaneswari A & Bhattacharjee K, 2026 (arXiv 2603.19656),
Tables 7–10 list the per-`t` equidistribution rank for the
combined 2-component CA-based PRNG with sizes `(k1, k2) = (31, 32)`
at time spacings `s ∈ {2, 4, 7, 8}`.  The paper claims:

- Table 7 (s = 2): **NOT** maximally equidistributed (4 of 13 rows equi)
- Table 8 (s = 4): **NOT** maximally equidistributed (7 of 13 rows equi)
- Table 9 (s = 7): **ME** (all 13 rows equi)
- Table 10 (s = 8): **ME** (all 13 rows equi)

This notebook reimplements the construction from scratch in plain
NumPy — no `regpoly`, no `regpoly_cpp` — and computes each rank under
the L'Écuyer / Proposition 2 definition the paper states in §2.3:
the matrix `B` has `t · l` rows from `t` advances of the recurrence,
and the sequence is `(t, l)`-equidistributed iff `rank(B) = t · l`.

## 1. The two component CAs

From paper **Example 1** (page 16):

```
R1 (k1 = 31) = ⟨90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 150,
                 90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90,
                 90, 90, 90, 90, 90, 90, 90, 90, 90⟩
          → rule-150 at cell index 11 (1-indexed) = 10 (0-indexed)

R2 (k2 = 32) = ⟨150, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90,
                 90, 90, 90, 150, 90, 90, 90, 90, 90, 90, 90,
                 90, 90, 90, 90, 90, 90, 90, 90, 90, 90⟩
          → rule-150 at cells 1 and 15 (1-indexed) = 0 and 14 (0-indexed)
```

Periods: `ρ₁ = 2³¹ − 1`, `ρ₂ = 2³² − 1`, `gcd(ρ₁, ρ₂) = 1`,
combined period `ρ = ρ₁ · ρ₂ ≈ 2⁶³`.

In [22]:
import numpy as np

K1, K2 = 31, 32
R1_R150 = [10]              # 0-indexed cell positions using rule 150
R2_R150 = [0, 14]
K_G = K1 + K2               # combined state size = 63

# Combined-generator convention: each component has its own output width
# L_i = min(k_i, word_size), and the combined output width is the minimum
# over components.  For (31, 32) with a 64-bit machine word this gives
# L = min(31, 32) = 31.
WORD_SIZE = 64
L1 = min(K1, WORD_SIZE)
L2 = min(K2, WORD_SIZE)
L_WORD = min(L1, L2)

print(f'K1 = {K1}, R1 rule-150 positions = {R1_R150}')
print(f'K2 = {K2}, R2 rule-150 positions = {R2_R150}')
print(f'k_g = K1 + K2 = {K_G}')
print(f'L_1 = min(K1, word_size) = {L1}')
print(f'L_2 = min(K2, word_size) = {L2}')
print(f'combined L = min(L_1, L_2) = {L_WORD}')

K1 = 31, R1 rule-150 positions = [10]
K2 = 32, R2 rule-150 positions = [0, 14]
k_g = K1 + K2 = 63
L_1 = min(K1, word_size) = 31
L_2 = min(K2, word_size) = 32
combined L = min(L_1, L_2) = 31


## 2. Build the transition matrices `T1`, `T2`

For a k-cell linear CA with null boundary the k×k transition matrix
is defined cell-by-cell:

- `T[i, i-1] = 1` if `i >= 1`        (cell `i` reads its left neighbour)
- `T[i, i+1] = 1` if `i <= k-2`      (cell `i` reads its right neighbour)
- `T[i, i]   = 1` iff cell `i` is rule 150 (otherwise rule 90, no self-term)

All arithmetic is over GF(2).

In [23]:
def build_T(k, rule150_positions):
    T = np.zeros((k, k), dtype=np.uint8)
    rset = set(rule150_positions)
    for i in range(k):
        if i - 1 >= 0:
            T[i, i - 1] = 1
        if i + 1 < k:
            T[i, i + 1] = 1
        if i in rset:
            T[i, i] = 1
    return T

T1 = build_T(K1, R1_R150)
T2 = build_T(K2, R2_R150)
print('T1 shape:', T1.shape, '  T2 shape:', T2.shape)



T1 shape: (31, 31)   T2 shape: (32, 32)


In [24]:
print('\nT1 (31×31):')
print(T1)


T1 (31×31):
[[0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 

In [25]:
print('\nT2  (32×32):')
print(T2)



T2  (32×32):
[[1 1 0 ... 0 0 0]
 [1 0 1 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 1 0 1]
 [0 0 0 ... 0 1 0]]


## 3. GF(2) rank and a sanity check on `T1`, `T2`

A primitive characteristic polynomial implies `T` is invertible over
GF(2), hence `rank(T) = k`.  We pin this for both components.

In [26]:
def gf2_rank(A):
    """Rank of a 0/1 matrix over GF(2) via Gaussian elimination."""
    M = A.astype(np.uint8).copy()
    nrows, ncols = M.shape
    r = 0
    for c in range(ncols):
        pivots = np.flatnonzero(M[r:, c])
        if pivots.size == 0:
            continue
        p = r + int(pivots[0])
        if p != r:
            M[[r, p]] = M[[p, r]]
        mask = M[:, c].copy()
        mask[r] = 0
        if mask.any():
            M[mask == 1] ^= M[r]
        r += 1
        if r == nrows:
            break
    return r

print('rank(T1) =', gf2_rank(T1), '  (expected 31)')
print('rank(T2) =', gf2_rank(T2), '  (expected 32)')

rank(T1) = 31   (expected 31)
rank(T2) = 32   (expected 32)


## 4. Time spacing: compute `T^s` over GF(2)

With time spacing `s`, the per-step transition is `T^s` (matrix power
over GF(2)).  We use repeated squaring.

In [27]:
def gf2_matmul(A, B):
    return ((A.astype(np.uint8) @ B.astype(np.uint8)) & 1).astype(np.uint8)

def gf2_matpow(T, s):
    """T^s over GF(2) by binary exponentiation."""
    k = T.shape[0]
    R = np.eye(k, dtype=np.uint8)
    base = T.astype(np.uint8).copy()
    e = s
    while e > 0:
        if e & 1:
            R = gf2_matmul(R, base)
        base = gf2_matmul(base, base)
        e >>= 1
    return R

S_VALUES = [2, 4, 7, 8]
T1_pow = {s: gf2_matpow(T1, s) for s in S_VALUES}
T2_pow = {s: gf2_matpow(T2, s) for s in S_VALUES}

for s in S_VALUES:
    print(f's = {s}:  rank(T1^s) = {gf2_rank(T1_pow[s])}, '
          f'rank(T2^s) = {gf2_rank(T2_pow[s])}  (both should equal k)')

s = 2:  rank(T1^s) = 31, rank(T2^s) = 32  (both should equal k)
s = 4:  rank(T1^s) = 31, rank(T2^s) = 32  (both should equal k)
s = 7:  rank(T1^s) = 31, rank(T2^s) = 32  (both should equal k)
s = 8:  rank(T1^s) = 31, rank(T2^s) = 32  (both should equal k)


## 5. The equidistribution matrix `B`

Following §2.3 / Proposition 2:

- Total state size `k_g = k1 + k2 = 63`.  Columns of `B` are indexed by
  the `k_g` unit-vector initial conditions of the combined state.
- For column `j < k1`: component 1 starts at `e_j`, component 2 at `0`.
  For column `j ≥ k1`: component 1 starts at `0`, component 2 at `e_{j-k1}`.
- Advance both components by `T1^s`, `T2^s` for `n = 1, …, t` steps.
- After each step `n`, the combined output bit at position `b` is
  `state1[b] XOR state2[b]` with 0-padding when `b ≥ k_i`.
- Stack the first `l` output bits at each of the `t` steps to form
  column `j` (`t · l` rows in total).

`(t, l)`-equidistributed iff `rank(B) = t · l`.

In [28]:
def build_B(T1s, T2s, k1, k2, t, l):
    """Build the (t·l) × (k1+k2) equidistribution matrix B.
    Step n ∈ {1, …, t}: advance, then record the first l output bits
    (state1[b] XOR state2[b], 0-padded when b ≥ k_i).
    """
    k_g = k1 + k2
    B = np.zeros((t * l, k_g), dtype=np.uint8)
    for j in range(k_g):
        s1 = np.zeros(k1, dtype=np.uint8)
        s2 = np.zeros(k2, dtype=np.uint8)
        if j < k1:
            s1[j] = 1
        else:
            s2[j - k1] = 1
        for n in range(t):
            s1 = (T1s @ s1) & 1
            s2 = (T2s @ s2) & 1
            for b in range(l):
                bit1 = int(s1[b]) if b < k1 else 0
                bit2 = int(s2[b]) if b < k2 else 0
                B[n * l + b, j] = bit1 ^ bit2
    return B

# Smoke test at (s, t, l) = (2, 2, 31): matrix should be 62 × 63.
B_smoke = build_B(T1_pow[2], T2_pow[2], K1, K2, t=2, l=31)
print('B smoke shape:', B_smoke.shape, '  (expected (62, 63))')

B smoke shape: (62, 63)   (expected (62, 63))


## 6. The dimension set `Φ₁ ∪ Φ₂` (Proposition 4)

For `k_g = 63` and combined `L = min(L_1, L_2) = 31`:

$$
\Phi_1 = \big\{\max(2, \lfloor k_g/L \rfloor), \ldots, \lfloor\sqrt{k_g}\rfloor\big\}, \qquad
\Phi_2 = \big\{\lfloor k_g/\ell \rfloor : \ell = 1, \ldots, \lfloor\sqrt{k_g}\rfloor\big\}
$$

In [29]:
import math
def phi_set(k_g, L):
    sq = int(math.isqrt(k_g))
    m  = max(2, k_g // L)
    phi1 = set(range(m, sq + 1))
    phi2 = {k_g // l for l in range(1, sq + 1)}
    return sorted(phi1 | phi2)

PHI = phi_set(K_G, L_WORD)
print('Φ₁ ∪ Φ₂ =', PHI)
print('size =', len(PHI), '(paper Tables 7–10 have 13 rows each)')

Φ₁ ∪ Φ₂ = [2, 3, 4, 5, 6, 7, 9, 10, 12, 15, 21, 31, 63]
size = 13 (paper Tables 7–10 have 13 rows each)


## 7. Driver: per-`t` rank for one `s`

For each `t ∈ Φ`, builds `B(t, l*)`, computes `rank(B)`, and prints
the verdict next to the paper's published values.

In [30]:
def run_table(s, paper_rows):
    T1s, T2s = T1_pow[s], T2_pow[s]
    print(f'\nCombined CA (31, 32) at s = {s}')
    print(f'{"t":>4} {"l*":>4} {"t·l":>5}  '
          f'{"rank":>5}  {"paper rank":>11}  '
          f'{"verdict":>10}  {"paper verdict":>14}')
    print('-' * 70)
    n_equi = 0
    for (t, lp, pr, pv) in paper_rows:
        l_star = min(L_WORD, K_G // t)
        assert l_star == lp, (t, l_star, lp)
        B = build_B(T1s, T2s, K1, K2, t, l_star)
        rk = gf2_rank(B)
        eq = (rk == t * l_star)
        n_equi += int(eq)
        print(f'{t:>4} {l_star:>4} {t*l_star:>5}  '
              f'{rk:>5}  {pr:>11}  '
              f'{("equi" if eq else "NOT equi"):>10}  '
              f'{("equi" if pv else "NOT equi"):>14}')
    print(f'\nME (all rows equi)?  {n_equi == len(paper_rows)}'
          f'   ({n_equi}/{len(paper_rows)} equi rows)')
    print(f'paper claim ME?      {all(v for *_, v in paper_rows)}')

## 8. Table 7 — `s = 2` (paper: NOT ME)

In [31]:
# (t, l*, paper_rank, paper_equi?)
PAPER_TABLE_7 = [
    (2,  31, 47, False), (3,  21, 46, False), (4,  15, 41, False),
    (5,  12, 38, False), (6,  10, 36, False), (7,   9, 37, False),
    (9,   7, 43, False), (10,  6, 46, False), (12,  5, 53, False),
    (15,  4, 60, True ), (21,  3, 63, True ), (31,  2, 62, True ),
    (63,  1, 63, True ),
]
run_table(s=2, paper_rows=PAPER_TABLE_7)


Combined CA (31, 32) at s = 2
   t   l*   t·l   rank   paper rank     verdict   paper verdict
----------------------------------------------------------------------
   2   31    62     39           47    NOT equi        NOT equi
   3   21    63     37           46    NOT equi        NOT equi
   4   15    60     33           41    NOT equi        NOT equi
   5   12    60     32           38    NOT equi        NOT equi
   6   10    60     30           36    NOT equi        NOT equi
   7    9    63     33           37    NOT equi        NOT equi
   9    7    63     39           43    NOT equi        NOT equi
  10    6    60     42           46    NOT equi        NOT equi
  12    5    60     49           53    NOT equi        NOT equi
  15    4    60     60           60        equi            equi
  21    3    63     63           63        equi            equi
  31    2    62     62           62        equi            equi
  63    1    63     63           63        equi            equi

M

## 9. Table 8 — `s = 4` (paper: NOT ME)

In [32]:
PAPER_TABLE_8 = [
    (2,  31, 60, False), (3,  21, 59, False), (4,  15, 57, False),
    (5,  12, 54, False), (6,  10, 58, False), (7,   9, 62, False),
    (9,   7, 63, True ), (10,  6, 60, True ), (12,  5, 60, True ),
    (15,  4, 60, True ), (21,  3, 63, True ), (31,  2, 62, True ),
    (63,  1, 63, True ),
]
run_table(s=4, paper_rows=PAPER_TABLE_8)


Combined CA (31, 32) at s = 4
   t   l*   t·l   rank   paper rank     verdict   paper verdict
----------------------------------------------------------------------
   2   31    62     47           60    NOT equi        NOT equi
   3   21    63     51           59    NOT equi        NOT equi
   4   15    60     49           57    NOT equi        NOT equi
   5   12    60     46           54    NOT equi        NOT equi
   6   10    60     50           58    NOT equi        NOT equi
   7    9    63     57           62    NOT equi        NOT equi
   9    7    63     62           63    NOT equi            equi
  10    6    60     59           60    NOT equi            equi
  12    5    60     60           60        equi            equi
  15    4    60     60           60        equi            equi
  21    3    63     62           63    NOT equi            equi
  31    2    62     62           62        equi            equi
  63    1    63     63           63        equi            equi

M

## 10. Table 9 — `s = 7` (paper: ME)

In [33]:
PAPER_TABLE_9 = [
    (2,  31, 62, True), (3,  21, 63, True), (4,  15, 60, True),
    (5,  12, 60, True), (6,  10, 60, True), (7,   9, 63, True),
    (9,   7, 63, True), (10,  6, 60, True), (12,  5, 60, True),
    (15,  4, 60, True), (21,  3, 63, True), (31,  2, 62, True),
    (63,  1, 63, True),
]
run_table(s=7, paper_rows=PAPER_TABLE_9)


Combined CA (31, 32) at s = 7
   t   l*   t·l   rank   paper rank     verdict   paper verdict
----------------------------------------------------------------------
   2   31    62     57           62    NOT equi            equi
   3   21    63     58           63    NOT equi            equi
   4   15    60     60           60        equi            equi
   5   12    60     60           60        equi            equi
   6   10    60     60           60        equi            equi
   7    9    63     62           63    NOT equi            equi
   9    7    63     62           63    NOT equi            equi
  10    6    60     60           60        equi            equi
  12    5    60     60           60        equi            equi
  15    4    60     60           60        equi            equi
  21    3    63     61           63    NOT equi            equi
  31    2    62     62           62        equi            equi
  63    1    63     63           63        equi            equi

M

## 11. Table 10 — `s = 8` (paper: ME)

In [34]:
PAPER_TABLE_10 = [
    (2,  31, 62, True), (3,  21, 63, True), (4,  15, 60, True),
    (5,  12, 60, True), (6,  10, 60, True), (7,   9, 63, True),
    (9,   7, 63, True), (10,  6, 60, True), (12,  5, 60, True),
    (15,  4, 60, True), (21,  3, 63, True), (31,  2, 62, True),
    (63,  1, 63, True),
]
run_table(s=8, paper_rows=PAPER_TABLE_10)


Combined CA (31, 32) at s = 8
   t   l*   t·l   rank   paper rank     verdict   paper verdict
----------------------------------------------------------------------
   2   31    62     59           62    NOT equi            equi
   3   21    63     61           63    NOT equi            equi
   4   15    60     60           60        equi            equi
   5   12    60     60           60        equi            equi
   6   10    60     60           60        equi            equi
   7    9    63     61           63    NOT equi            equi
   9    7    63     61           63    NOT equi            equi
  10    6    60     60           60        equi            equi
  12    5    60     60           60        equi            equi
  15    4    60     60           60        equi            equi
  21    3    63     62           63    NOT equi            equi
  31    2    62     62           62        equi            equi
  63    1    63     63           63        equi            equi

M

## 12. Summary

Under the L'Écuyer / Proposition 2 definition the paper itself states
in §2.3 (matrix `B` with `t · l` rows, `(t, l)`-equi iff `rank(B) = t · l`):

| s | per-row equi count | overall verdict | paper claim |
|---|---|---|---|
| 2 | 4 / 13 | NOT ME | NOT ME — agrees |
| 4 | 4 / 13 | NOT ME | NOT ME — agrees |
| 7 | 8 / 13 | NOT ME | ME — disagrees |
| 8 | 8 / 13 | NOT ME | ME — disagrees |

The per-row rank values printed above are also systematically lower
than the paper's published ranks at every deficient row.  Both the
rank values and the overall ME verdicts for `s = 7` and `s = 8` differ
from the paper — that is, the paper's two ME claims for (31, 32) do
not hold under the procedure the paper itself describes.